# LifeMate: bước 3, sinh dữ liệu tổng hợp bằng Claude

Với mỗi mẫu train, Claude đọc context, history và dữ liệu các app của mẫu đó rồi viết yêu cầu nói mới kèm chuỗi tool đúng: một yêu cầu làm được, một yêu cầu nhiều việc hoặc có điều kiện, một yêu cầu thiếu thông tin (phải hỏi lại) và bản đủ thông tin của chính yêu cầu đó, thỉnh thoảng thêm một yêu cầu ngoài phạm vi. Mỗi chuỗi được chạy thật trên `ToolEnv` của mẫu gốc và bị bỏ nếu có call lỗi, nếu dùng ID mà tool đọc chưa trả về, hoặc nếu sai loại.

Chuẩn bị:
1. **Runtime: CPU là đủ** (chỉ gọi API).
2. Secret **`ANTHROPIC_API_KEY`** trong Colab Secrets (biểu tượng chìa khoá bên trái), bật quyền cho notebook. Không dán key vào cell.
3. Trên Drive đã có `MyDrive/zalo_lifemate/train+public_test.zip`, secret `GH_TOKEN` như trước.

Kết quả ghi thẳng lên Drive `MyDrive/zalo_lifemate/synth/synth.jsonl` và chạy tiếp được nếu bị ngắt: chạy lại cell thì nó bỏ qua mẫu đã làm. Dòng `[synth] ... cost~$` cho biết tiền đã tiêu (ước tính).

Luật cho phép dùng LLM ngoài để sinh dữ liệu trong lúc phát triển, không dùng lúc chạy bài thi. Notebook này gửi context, history và dữ liệu app của các mẫu **train** lên API của Anthropic; không gửi gì của public_test.

In [ ]:
import os, json
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
os.environ["PYTHONPATH"] = f"{REPO}/src"
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
SYNTH = f"{DRIVE}/synth/synth.jsonl"
!mkdir -p "{DRIVE}/synth"
!pip install -q anthropic
# kiểm tra bộ lọc trên chính đáp án train (không gọi API): gần như mọi mẫu phải qua
!python scripts/synth_data.py --data ./train+public_test --out /dev/null --check-gold

## 1. Chạy thử 20 mẫu (khoảng 3 USD với Opus 5.5)
`MODEL` mặc định là `claude-opus-5-5` (tốt nhất). Muốn rẻ hơn thì đổi trước khi chạy: `claude-sonnet-5-5` khoảng một nửa giá, `claude-haiku-5-5` rẻ hơn nhiều nhưng dữ liệu kém hơn. Nên giữ một model cho cả lần sinh.

Xong cell này thì **tải `synth.jsonl` và `synth.jsonl.done` trong `MyDrive/zalo_lifemate/synth/` về `runs/` trên máy và báo mình xem chất lượng** trước khi chạy hết.

In [ ]:
MODEL, EFFORT, WORKERS = "claude-opus-5-5", "medium", 4
!python scripts/synth_data.py --data ./train+public_test --ids-file splits/train2_ids.txt --out "{SYNTH}" --model {MODEL} --effort {EFFORT} --workers {WORKERS} --limit 20

In [ ]:
# xem nhanh: số mẫu theo loại, lý do bị bỏ, và vài mẫu
from collections import Counter
rows = [json.loads(l) for l in open(SYNTH)]
done = [json.loads(l) for l in open(SYNTH + ".done")]
print(len(rows), "mẫu giữ lại từ", len(done), "mẫu gốc:", Counter(r["synth_type"] for r in rows))
print("bị bỏ:", Counter(w for d in done for w in d["rejected"]).most_common(10))
for r in rows[:8]:
    print(f"\n[{r['synth_type']}] {r['audio_text']}")
    for c in r["label"]["tool_calls"]:
        print("   ", c["name"], json.dumps(c["arguments"], ensure_ascii=False))

## 2. Chạy hết các mẫu train2 (cho thí nghiệm trên dev2)
1245 mẫu gốc. Với Opus 5.5 ước tính khoảng 150 USD và 3–5 giờ với 4 luồng; số tiền thật xem ở dòng `cost~$` sau lần chạy thử. Tăng `WORKERS` nếu tài khoản cho phép nhiều request hơn (lỗi 429 thì giảm). Bị ngắt thì chạy lại cell này.

In [ ]:
!python scripts/synth_data.py --data ./train+public_test --ids-file splits/train2_ids.txt --out "{SYNTH}" --model {MODEL} --effort {EFFORT} --workers {WORKERS}

## 3. Sau này: các mẫu dev2 (chỉ cho model cuối cùng)
300 mẫu nữa, chỉ dùng khi train bản cuối trên toàn bộ train. Khi chấm dev2, `build_sft.py --ids-file splits/train2_ids.txt` tự bỏ các mẫu sinh từ dev2 nên chạy cell này sớm cũng không làm hỏng phép đo.

In [ ]:
!python scripts/synth_data.py --data ./train+public_test --ids-file splits/dev2_ids.txt --out "{SYNTH}" --model {MODEL} --effort {EFFORT} --workers {WORKERS}